In [ ]:
import torch
import torch.nn as nn
import numpy as np
import random

from kan import KAN
from kan.utils import create_dataset

import wandb
# import torch.profiler
# import fvcore
import time


import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device, ' '+torch.__version__)

In [ ]:
# Define target function from KAN paper
def target_func(x):
    return torch.exp(torch.sin(torch.pi * x[:,0]) + x[:, 1]**2)

In [ ]:
# Create a 2D grid of inputs
grid_size = 1000  # 100x100 grid = 10,000 points
x_vals = torch.linspace(-1, 1, grid_size)
y_vals = torch.linspace(-1, 1, grid_size)
X, Y = torch.meshgrid(x_vals, y_vals, indexing='ij')

# Flatten the grid and pass it to the target function
grid_input = torch.stack([X.flatten(), Y.flatten()], dim=1)
# print(grid_input)

# Assume target_func takes a [N, 2] input and returns a [N] or [N, 1] output
Z = target_func(grid_input).reshape(grid_size, grid_size)

# Plot the surface
fig = plt.figure(figsize=(16, 9))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X.numpy(), Y.numpy(), Z.numpy(), label = 'True', alpha = 0.5)

from matplotlib.lines import Line2D

legend_proxy = Line2D([0], [0], marker='s', color='w', label='True',
                      markerfacecolor='green', markersize=10, alpha=0.5)

ax.set_xlabel('Input X')
ax.set_ylabel('Input Y')
ax.set_zlabel('Label Z')
ax.legend(handles= [legend_proxy])
plt.show()


In [ ]:
def count_parameters(model):
    """Count trainable parameters in a PyTorch model"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
# Create dataset
n_var = 2
ranges = [-1, 1]
train_num: int = 10000
test_num: int = 10000
dataset = create_dataset(target_func, n_var=n_var, ranges=ranges, train_num=train_num, test_num=test_num, device=device)

In [ ]:
# Initialize W&B project
wandb.init(
    entity="hpml_project_spring25",
    project="KAN-Quantization-testing",
    config={
    "seed": random.randint(0, 200),
    "grid_size": 5,
    "epochs": 100,
    "lr": 0.01,
    # "batch_size": 256,
    "equation": "exp(sin(\pi x) + x**2)",
    "n_var": n_var,
    "ranges": ranges,
    "train_num": train_num,
    "test_num": test_num,
    "input_dim": n_var,
    "kan_hidden_dim": 5,
    "mlp_hidden_dim": None,
    "output_dim": 1,
    "spline_order":3, 
})

In [ ]:
# Initialize models
# from fvcore.nn import FlopCountAnalysis

model = KAN(
    width=[wandb.config.input_dim, wandb.config.kan_hidden_dim, wandb.config.output_dim], 
    grid=wandb.config.grid_size, 
    k=wandb.config.spline_order, 
    seed=wandb.config.seed, 
    device=device)
kan_params = count_parameters(model)

wandb.config.kan_params = kan_params
# wandb.config.mlp_params = mlp_params


# Track gradients and parameters
# wandb.watch((kan, mlp), log="all")

# Loss and optimizers
criterion = torch.nn.MSELoss()
kan_optim = torch.optim.Adam(model.parameters(), lr=wandb.config.lr)

grid_size = wandb.config.grid_size 

In [ ]:
from kan.quantization import QuantizedKAN

In [ ]:
#prepare model to get quantized
quantized_model = QuantizedKAN(model)
quantized_model.qconfig = torch.quantization.get_default_qconfig('fbgemm')
torch.quantization.prepare(quantized_model, inplace=True)

In [ ]:
dummy_input = torch.randn(100, 2, device=device)  # Batch size 100, 2 features
# forward pass just to populate activation functions
quantized_model(dummy_input)

In [ ]:
# Training loop
for epoch in range(wandb.config.epochs):
    quantized_model.train()

    # KAN training
    kan_optim.zero_grad()
    torch.cuda.synchronize()
    start_time = time.perf_counter()
    kan_pred = quantized_model(dataset['train_input'])
    kan_loss = criterion(kan_pred, dataset['train_label'])
    kan_loss.backward()
    torch.cuda.synchronize()
    duration_kan = (time.perf_counter() - start_time)
    kan_optim.step()

In [ ]:
torch.quantization.convert(quantized_model, inplace=True)

In [ ]:
# Step 7: Measure inference time
def benchmark(model, inputs, runs=1):
    model.eval()
    with torch.no_grad():
        start = time.time()
        for _ in range(runs):
            model(inputs)
        end = time.time()
    return (end - start) / runs * 1000  # ms per run

normal_time = benchmark(model, dataset['test_input'])
quant_time = benchmark(quantized_model, dataset['test_input'])
print(f'Without quantization: {normal_time} \n With quantization: {quant_time}')

In [ ]:
# Validation
with torch.no_grad():
    quantized_model.eval()
    kan_val = criterion(quantized_model(dataset['test_input']), dataset['test_label'])

In [ ]:
# Validation
with torch.no_grad():
    kan.eval()
    # mlp.eval()

    
    kan_val = criterion(kan(dataset['test_input']), dataset['test_label'])
    mlp_val = criterion(mlp(dataset['test_input']), dataset['test_label'])

    # flop_kan = FlopCountAnalysis(kan, dataset['train_input']).total()
    # flop_mlp = FlopCountAnalysis(mlp, dataset['train_input']).total()

    # print('flop duration mlp', flop_mlp, duration_mlp)
    # flops_per_sec_kan = flop_kan / duration_kan
    # flops_per_sec_mlp = flop_mlp / duration_mlp

# Log metrics
wandb.log({
    "epoch": epoch,
    "kan_train_loss": kan_loss.item(),
    #"mlp_train_loss": mlp_loss.item(),
    "kan_val_loss": kan_val.item(),
    "mlp_val_loss": mlp_val.item(),
    "kan_val_rmse": torch.sqrt(kan_val).item(),
    "mlp_val_rmse": torch.sqrt(mlp_val).item(),
    "kan_trainable_params": count_parameters(kan),
    "mlp_trainable_params": count_parameters(mlp),
    # "kan_flops": flop_kan,
    # "mlp_flops": flop_mlp,
    # "kan_flops_per_sec": flops_per_sec_kan,
    # "mlp_flops_per_sec": flops_per_sec_mlp,
    "kan_train_time_per_epoch_sec": duration_kan,
    #"mlp_train_time_per_epoch_sec": duration_mlp,
})

if (epoch + 1) % 10 == 0:
    wandb.config.grid_size += 5
    kan = kan.refine(wandb.config.grid_size)
    kan_optim = torch.optim.Adam(kan.parameters(), lr=wandb.config.lr)

    #mlp, _ = build_mlp(wandb.config.input_dim, 1, wandb.config.output_dim, kan, copy_mlp=mlp, seed=wandb.config.seed)
    mlp = mlp.to(device)
    mlp_optim = torch.optim.Adam(mlp.parameters(), lr=wandb.config.lr)

# Final evaluation
with torch.no_grad():
    kan.eval()
    mlp.eval()
    

    grid_plot_size = 100
    # Generate predictions
    x = torch.linspace(wandb.config.ranges[0], wandb.config.ranges[1], grid_plot_size)
    X, Y = torch.meshgrid(x, x, indexing='ij')
    grid_input = torch.stack([X.flatten(), Y.flatten()], dim=1).to(device)

    Z = target_func(grid_input).reshape(grid_plot_size, grid_plot_size)
    kan_pred = kan(grid_input).cpu().reshape(grid_plot_size, grid_plot_size)
    mlp_pred = mlp(grid_input).cpu().reshape(grid_plot_size, grid_plot_size)

    fig = plt.figure(figsize=(16, 9))
    ax = fig.add_subplot(111, projection='3d')
    # Plot surfaces
    ax.plot_surface(X.cpu().numpy(), Y.cpu().numpy(), Z.cpu().numpy(), alpha=0.5, color='cyan')
    ax.plot_surface(X.cpu().numpy(), Y.cpu().numpy(), kan_pred.cpu().numpy(), alpha=0.5, color='magenta')
    ax.plot_surface(X.cpu().numpy(), Y.cpu().numpy(), mlp_pred.cpu().numpy(), alpha=0.5, color='yellow')

    # Create proxy artists for legend
    legend_proxies = [
        Line2D([0], [0], marker='s', linestyle='none', markersize=10, 
            markerfacecolor='cyan', alpha=0.5, label='True'),
        Line2D([0], [0], marker='s', linestyle='none', markersize=10, 
            markerfacecolor='magenta', alpha=0.5, label='KAN'),
        Line2D([0], [0], marker='s', linestyle='none', markersize=10, 
            markerfacecolor='yellow', alpha=0.5, label='MLP')
    ]

    ax.set_xlabel('Input X')
    ax.set_ylabel('Input Y')
    ax.set_zlabel('Label Z')
    ax.legend(handles=legend_proxies)
    wandb.log({"predictions": wandb.Image(fig)})
    plt.close()

# Parametric complexity analysis
wandb.log({
    "kan_vs_mlp_complexity": wandb.plot.line_series(
        xs=[wandb.config.epochs, wandb.config.epochs],
        ys=[[sum(p.numel() for p in kan.parameters())], 
            [sum(p.numel() for p in mlp.parameters())]],
        title="Model Complexity",
        xname="Epochs",
        keys=["KAN", "MLP"]
    )
})

wandb.finish()